# Calendar Dimension Table

Builds the standard calendar dimension `adb_caresync.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.silver.calendar
COMMENT 'Standard calendar dimension: one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                 AS datekey,
  d                                                        AS date,
  -- Year
  year(d)                                                  AS year,
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END                AS half_year,
  CONCAT('H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END)   AS half_year_name,
  -- Quarter
  quarter(d)                                               AS quarter,
  CONCAT('Q', quarter(d))                                  AS quarter_name,
  CONCAT(year(d), '-Q', quarter(d))                        AS year_quarter,
  -- Month
  month(d)                                                 AS month,
  date_format(d, 'MMMM')                                   AS month_name,
  date_format(d, 'MMM')                                    AS month_short_name,
  date_format(d, 'yyyy-MM')                                AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                    AS year_month_key,
  -- Week (ISO 8601: weeks start Monday)
  weekofyear(d)                                            AS week,
  extract(YEAROFWEEK FROM d)                               AS iso_week_year,
  CONCAT(extract(YEAROFWEEK FROM d), '-W', lpad(weekofyear(d), 2, '0')) AS year_week,
  CAST(ceil(day(d) / 7.0) AS INT)                          AS week_of_month,
  -- Day
  day(d)                                                   AS day_of_month,
  dayofyear(d)                                             AS day_of_year,
  weekday(d) + 1                                           AS day_of_week,       -- 1=Monday ... 7=Sunday
  date_format(d, 'EEEE')                                   AS day_name,
  date_format(d, 'EEE')                                    AS day_short_name,
  -- Flags
  CASE WHEN weekday(d) >= 5 THEN true ELSE false END       AS is_weekend,
  CASE WHEN weekday(d) <  5 THEN true ELSE false END       AS is_weekday,
  (d = trunc(d, 'MONTH'))                                  AS is_month_start,
  (d = last_day(d))                                        AS is_month_end,
  (d = trunc(d, 'QUARTER'))                                AS is_quarter_start,
  (d = date_sub(add_months(trunc(d, 'QUARTER'), 3), 1))    AS is_quarter_end,
  (d = trunc(d, 'YEAR'))                                   AS is_year_start,
  (month(d) = 12 AND day(d) = 31)                          AS is_year_end,
  ((year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0) AS is_leap_year,
  -- Period boundaries
  CAST(date_trunc('WEEK', d) AS DATE)                      AS week_start_date,
  date_add(CAST(date_trunc('WEEK', d) AS DATE), 6)         AS week_end_date,
  trunc(d, 'MONTH')                                        AS month_start_date,
  last_day(d)                                              AS month_end_date,
  day(last_day(d))                                         AS days_in_month,
  trunc(d, 'QUARTER')                                      AS quarter_start_date,
  date_sub(add_months(trunc(d, 'QUARTER'), 3), 1)          AS quarter_end_date,
  trunc(d, 'YEAR')                                         AS year_start_date,
  make_date(year(d), 12, 31)                               AS year_end_date
FROM dates

In [0]:
%sql
SELECT
  COUNT(*)                         AS row_count,
  COUNT(DISTINCT datekey)          AS distinct_datekeys,
  MIN(date)                        AS min_date,
  MAX(date)                        AS max_date,
  datediff(MAX(date), MIN(date)) + 1 AS expected_days,
  SUM(CASE WHEN is_leap_year AND month = 2 AND day_of_month = 29 THEN 1 ELSE 0 END) AS feb29_count
FROM adb_caresync.silver.calendar

In [0]:
%sql
SELECT * FROM adb_caresync.silver.calendar
WHERE date BETWEEN DATE'2024-12-28' AND DATE'2025-01-03'
ORDER BY date